In [1]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score

In [2]:
# Select GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [3]:
# Load the pretrained ResNet18 architecture
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Replace the final classification layer for CIFAR-10 (10 classes)
model.fc = torch.nn.Linear(model.fc.in_features, 10)

# Move the model to the selected device (GPU/CPU)
model = model.to(device)

print("Model architecture created successfully!")

Model architecture created successfully!


In [4]:
# Path to the saved model
MODEL_PATH = "../models/resnet18_cifar10_phase2.pth"

# Load the trained weights
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))

# Set the model to evaluation mode
model.eval()

print("✅ Trained model loaded successfully!")

✅ Trained model loaded successfully!


In [5]:
# Transform for test images
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2023, 0.1994, 0.2010]
    )
])

print("✅ Test transform created successfully!")

✅ Test transform created successfully!


In [6]:
# Load the CIFAR-10 test dataset
test_dataset = datasets.CIFAR10(
    root="../data",
    train=False,
    download=True,
    transform=test_transform
)

print("Number of test images:", len(test_dataset))

Number of test images: 10000


c:\Reasearch Project\Semantic-communication\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [7]:
# Create the test DataLoader
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print("Number of test batches:", len(test_loader))

Number of test batches: 313


In [8]:
# Lists to store results
actual_labels = []
predicted_labels = []

# Disable gradient calculations (faster inference)
with torch.no_grad():

    for images, labels in test_loader:

        # Move data to GPU
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)

        # Get predicted class
        _, predictions = torch.max(outputs, 1)

        # Store labels
        actual_labels.extend(labels.cpu().numpy())
        predicted_labels.extend(predictions.cpu().numpy())

print("✅ Predictions completed!")
print("Total predictions:", len(predicted_labels))

✅ Predictions completed!
Total predictions: 10000


In [9]:
# Create a DataFrame with actual and predicted labels
results_df = pd.DataFrame({
    "Actual Label": actual_labels,
    "Predicted Label": predicted_labels
})

# Display the first 10 rows
results_df.head(10)

,Actual Label,Predicted Label
0,3,3
1,8,8
2,8,8
3,0,0
4,6,6
5,6,6
6,1,1
7,6,6
8,3,3
9,1,1


In [10]:
# Calculate test accuracy
accuracy = accuracy_score(actual_labels, predicted_labels)

print(f"Test Accuracy: {accuracy * 100:.2f}%")

Test Accuracy: 90.35%


In [13]:
from torchvision.transforms import GaussianBlur

gaussian_blur_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    GaussianBlur(kernel_size=5, sigma=2.0),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2023, 0.1994, 0.2010]
    )
])

In [14]:
# Load the corrupted CIFAR-10 test dataset
corrupted_test_dataset = datasets.CIFAR10(
    root="../data",
    train=False,
    download=False,
    transform=gaussian_blur_transform
)

print("Number of corrupted test images:", len(corrupted_test_dataset))

Number of corrupted test images: 10000


c:\Reasearch Project\Semantic-communication\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [15]:
# Create DataLoader for corrupted images
corrupted_test_loader = DataLoader(
    corrupted_test_dataset,
    batch_size=32,
    shuffle=False
)

print("Number of corrupted batches:", len(corrupted_test_loader))

Number of corrupted batches: 313


In [17]:
import torch.nn.functional as F

# Lists to store corrupted image results
corrupted_predictions = []
confidence_scores = []

# Disable gradients during inference
with torch.no_grad():

    for images, labels in corrupted_test_loader:

        # Move images to GPU
        images = images.to(device)

        # Model predictions
        outputs = model(images)

        # Convert logits to probabilities
        probabilities = F.softmax(outputs, dim=1)

        # Maximum probability = confidence
        confidence, predictions = torch.max(probabilities, dim=1)

        # Store results
        corrupted_predictions.extend(predictions.cpu().numpy())
        confidence_scores.extend(confidence.cpu().numpy())

print("✅ Corrupted image evaluation completed!")
print("Total corrupted predictions:", len(corrupted_predictions))
print("Total confidence scores:", len(confidence_scores))

✅ Corrupted image evaluation completed!
Total corrupted predictions: 10000
Total confidence scores: 10000


In [18]:
# Compute Task Consistency
task_consistency = []

for original_pred, corrupted_pred in zip(predicted_labels, corrupted_predictions):

    if original_pred == corrupted_pred:
        task_consistency.append(1)
    else:
        task_consistency.append(0)

# Average Task Consistency
task_consistency_score = sum(task_consistency) / len(task_consistency)

print(f"Task Consistency: {task_consistency_score:.4f}")

Task Consistency: 0.9832


In [19]:
# Compute TaskGT (Task Success)
task_gt = []

for ground_truth, corrupted_pred in zip(actual_labels, corrupted_predictions):

    if ground_truth == corrupted_pred:
        task_gt.append(1)
    else:
        task_gt.append(0)

# Average TaskGT
task_gt_score = sum(task_gt) / len(task_gt)

print(f"TaskGT (Task Success): {task_gt_score:.4f}")

TaskGT (Task Success): 0.9007


In [20]:
# Compute average confidence
average_confidence = sum(confidence_scores) / len(confidence_scores)

print(f"Average Confidence: {average_confidence:.4f}")

Average Confidence: 0.9327


In [22]:
%pip install transformers

  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
    --------------------------------------- 0.3/11.6 MB ? eta -:--:--
    --------------------------------------- 0.3/11.6 MB ? eta -:--:--
    --------------------------------------- 0.3/11.6 MB ? eta -:--:--
    --------------------------------------- 0.3/11.6 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.6 MB 344.2 kB/s eta 0:00:33
   - -------------------------------------- 0.5/11.6 MB 344.2 kB/s eta 0:00:33
   - -------------------------------------- 0.5/11.6 MB 344.2 kB/s eta 0:00:33
   -- ------------------------------------- 0.8/11.6 MB 366.0 kB/s eta 0:00:30
   -- ------------------------------------- 0.8/11.6 MB 366.0 kB/s eta 0:00:30
   -- ------------------------------------- 0.8/11.6 MB 366.0 kB/s eta 0:00:30
   --- ------------------------------------ 

In [1]:
from transformers import AutoImageProcessor, AutoModel

print("✅ Transformers imported successfully!")

✅ Transformers imported successfully!


In [1]:
from transformers import AutoImageProcessor, AutoModel

MODEL_NAME = "facebook/dinov2-small"

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
dino_model = AutoModel.from_pretrained(MODEL_NAME)

dino_model = dino_model.to(device)
dino_model.eval()

print("✅ DINOv2 loaded successfully!")

preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

c:\Reasearch Project\Semantic-communication\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\G Anushree\.cache\huggingface\hub\models--facebook--dinov2-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

KeyboardInterrupt: 